In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time


In [21]:
BASE_URL = "https://www.gsmarena.com/"
BRAND_URL = "https://www.gsmarena.com/samsung-phones-9.php"  # contoh: Samsung

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

all_phones = []

In [9]:
# Daftar brand (Nama → URL page pertama)
BRANDS = {
    "Samsung": "https://www.gsmarena.com/samsung-phones-9.php",
    "Xiaomi": "https://www.gsmarena.com/xiaomi-phones-80.php",
    "Infinix": "https://www.gsmarena.com/infinix-phones-119.php",
    "Apple": "https://www.gsmarena.com/apple-phones-48.php",
    "Oppo": "https://www.gsmarena.com/oppo-phones-82.php",
    "Vivo": "https://www.gsmarena.com/vivo-phones-98.php"
}

all_phones = []


In [22]:
def scrape_phone_detail(detail_url):
    """Scrape detail spesifikasi hanya atribut yang dibutuhkan"""
    res = requests.get(detail_url, headers=headers)
    soup = BeautifulSoup(res.text, "html.parser")

    # Nama HP (Merk + Seri)
    name = soup.find("h1", class_="specs-phone-name-title").text.strip()

    # Tabel spesifikasi
    specs_table = soup.find("div", id="specs-list")
    specs = {"Name": name, "URL": detail_url}

    # Released date (Launch - Announced)
    try:
        specs["Released"] = specs_table.find("td", text="Announced").find_next("td").text.strip()
    except:
        specs["Released"] = None

    # Berat (Body - Weight)
    try:
        specs["Weight"] = specs_table.find("td", text="Weight").find_next("td").text.strip()
    except:
        specs["Weight"] = None

    # Thickness (Body - Dimensions -> ambil nilai tebal)
    try:
        dims = specs_table.find("td", text="Dimensions").find_next("td").text.strip()
        specs["Thickness"] = dims.split()[-2] + " " + dims.split()[-1] if "mm" in dims else dims
    except:
        specs["Thickness"] = None

    # RAM & Storage (Memory - Internal)
    try:
        memory = specs_table.find("td", text="Internal").find_next("td").text.strip()
        specs["Memory"] = memory
        # Pisahkan RAM dan ROM kalau ada
        if "GB" in memory:
            parts = memory.split(",")[0]  # ambil varian pertama
            specs["RAM"] = [p for p in parts.split() if "GB" in p][0] if any("GB" in p for p in parts.split()) else None
            specs["Storage"] = parts
        else:
            specs["RAM"], specs["Storage"] = None, None
    except:
        specs["Memory"], specs["RAM"], specs["Storage"] = None, None, None

    # Kamera utama (Main Camera - Single/Dual/Triple/Quad)
    try:
        cam = specs_table.find("td", text="Triple").find_next("td").text.strip()
    except:
        try:
            cam = specs_table.find("td", text="Dual").find_next("td").text.strip()
        except:
            try:
                cam = specs_table.find("td", text="Single").find_next("td").text.strip()
            except:
                cam = None
    specs["Camera"] = cam

    # Harga (Misc - Price)
    try:
        specs["Price"] = specs_table.find("td", text="Price").find_next("td").text.strip()
    except:
        specs["Price"] = None

    return specs

In [23]:
def scrape_listing(listing_url):
    """Scrape daftar HP dari halaman listing brand"""
    res = requests.get(listing_url, headers=headers)
    soup = BeautifulSoup(res.text, "html.parser")
    phones = soup.find("div", class_="makers").find_all("li")
    phone_links = [BASE_URL + p.find("a")["href"] for p in phones]
    return phone_links

In [24]:
# Scrape daftar produk brand (Samsung contoh)
phone_links = scrape_listing(BRAND_URL)

for idx, link in enumerate(phone_links[:10], start=1):  # contoh: ambil 10 HP pertama
    print(f"Scraping {idx}: {link}")
    try:
        phone_data = scrape_phone_detail(link)
        all_phones.append(phone_data)
    except Exception as e:
        print(f"❌ Gagal scrap {link} -> {e}")
    time.sleep(2)

AttributeError: 'NoneType' object has no attribute 'find_all'

In [12]:
# Loop semua brand
for brand, url in BRANDS.items():
    print(f"\n📱 Scraping brand: {brand}")
    phone_links = scrape_listing(url)

    for idx, link in enumerate(phone_links, start=1):
        print(f"   {idx}/{len(phone_links)}: {link}")
        try:
            phone_data = scrape_phone_detail(link, brand)
            all_phones.append(phone_data)
        except Exception as e:
            print(f"❌ Gagal scrap {link} -> {e}")
        time.sleep(1)  # biar tidak diblok


📱 Scraping brand: Samsung
   🔎 Fetch listing: https://www.gsmarena.com/samsung-phones-9.php

📱 Scraping brand: Xiaomi
   🔎 Fetch listing: https://www.gsmarena.com/xiaomi-phones-80.php

📱 Scraping brand: Infinix
   🔎 Fetch listing: https://www.gsmarena.com/infinix-phones-119.php

📱 Scraping brand: Apple
   🔎 Fetch listing: https://www.gsmarena.com/apple-phones-48.php

📱 Scraping brand: Oppo
   🔎 Fetch listing: https://www.gsmarena.com/oppo-phones-82.php

📱 Scraping brand: Vivo
   🔎 Fetch listing: https://www.gsmarena.com/vivo-phones-98.php


In [8]:
# Simpan ke CSV
df = pd.DataFrame(ALL_PHONES)
df.to_csv("[local file]", index=False, encoding="utf-8-sig")

print("✅ Selesai! Data tersimpan di gsmarena_samsung.csv")


✅ Selesai! Data tersimpan di gsmarena_samsung.csv
